# Example 8.9 / 1: Compare Simplex and Barrier with HiGHS

Companion notebook for **8.9 How LP Solvers Work** in the AMPL lab Documentation, section "Compare Simplex and Barrier with HiGHS". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Compare Simplex and Barrier with HiGHS

The HiGHS option `alg:method` selects the LP algorithm, and `bar:crossover` controls crossover
after an interior-point run. This example solves the steel model three times: by simplex, by
interior point with crossover, and by interior point without crossover. In `data.py` and
`run.py`, `ampl` is an `AMPL` object that has already read `model.mod`.

HiGHS's own presolve reduces this tiny LP so far that the log reports 0 iterations of every
algorithm. The example adds `pre:solve=off` so that each algorithm does visible work.

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set PROD;                                 # products
param rate {PROD} > 0;                    # tons produced per hour
param avail >= 0;                         # hours available
param profit {PROD};                      # profit per ton
param market {PROD} >= 0;                 # most tons that can be sold
var Make {p in PROD} >= 0, <= market[p];  # tons produced, up to the market limit
maximize Total_Profit: sum {p in PROD} profit[p] * Make[p];  # total profit
# production hours fit the hours available
subject to Time: sum {p in PROD} (1 / rate[p]) * Make[p] <= avail;

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
ampl.set["PROD"] = ["bands", "coils", "plate"]
ampl.param["rate"] = {"bands": 200, "coils": 140, "plate": 160}
ampl.param["profit"] = {"bands": 25, "coils": 30, "plate": 29}
ampl.param["market"] = {"bands": 6000, "coils": 4000, "plate": 3500}
ampl.param["avail"] = 40

In [5]:
# run.py
for opts in ["alg:method=simplex", "alg:method=ipm", "alg:method=ipm bar:crossover=off"]:
    ampl.solve(solver="highs", highs_options=opts + " pre:solve=off")
    print(ampl.solve_result, "Time:", ampl.get_value("Time.sstatus"))
    print(ampl.get_data("Make, Make.sstatus").to_pandas())

HiGHS 1.14.0:   lp:method = simplex
  pre:solve = off
HiGHS 1.14.0: optimal solution; objective 196400
1 simplex iterations
0 barrier iterations
solved Time: upp
       Make Make.sstatus
bands  6000          upp
coils     0          low
plate  1600          bas
HiGHS 1.14.0:   lp:method = ipm
  pre:solve = off
HiGHS 1.14.0: optimal solution; objective 196400
0 simplex iterations
6 barrier iterations
solved Time: upp
       Make Make.sstatus
bands  6000          upp
coils     0          low
plate  1600          bas
HiGHS 1.14.0:   lp:method = ipm
  bar:crossover = off
  pre:solve = off
HiGHS 1.14.0: optimal solution; objective 196399.9996
0 simplex iterations
6 barrier iterations
solved Time: bas
              Make Make.sstatus
bands  5999.999925          sup
coils     0.000043          sup
plate  1600.000006          sup


### Pure AMPL: data.dat and model.run

`ampl.solve(..., highs_options=...)` sets the driver options for each run,
`ampl.solve_result` gives the status, `ampl.get_value("Time.sstatus")` reads one constraint's
status, and `ampl.get_data("Make, Make.sstatus").to_pandas()` returns values with statuses.

The same in pure AMPL, with the same `model.mod`:

In [6]:
%%writefile data.dat
set PROD := bands coils plate;
param rate := bands 200  coils 140  plate 160;
param profit := bands 25  coils 30  plate 29;
param market := bands 6000  coils 4000  plate 3500;
param avail := 40;

Writing data.dat


In [7]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
option solver highs;
option display_precision 10;
for {opts in {"alg:method=simplex", "alg:method=ipm", "alg:method=ipm bar:crossover=off"}} {
    option highs_options (opts & " pre:solve=off");
    solve;
    display solve_result, Time.sstatus;
    display Make, Make.sstatus;
}

HiGHS 1.14.0:   lp:method = simplex
  pre:solve = off
HiGHS 1.14.0: optimal solution; objective 196400
1 simplex iterations
0 barrier iterations
solve_result = solved
Time.sstatus = upp

:       Make Make.sstatus    :=
bands   6000   upp
coils      0   low
plate   1600   bas
;

HiGHS 1.14.0:   lp:method = ipm
  pre:solve = off
HiGHS 1.14.0: optimal solution; objective 196400
0 simplex iterations
6 barrier iterations
solve_result = solved
Time.sstatus = upp

:       Make Make.sstatus    :=
bands   6000   upp
coils      0   low
plate   1600   bas
;

HiGHS 1.14.0:   lp:method = ipm
  bar:crossover = off
  pre:solve = off
HiGHS 1.14.0: optimal solution; objective 196399.9996
0 simplex iterations
6 barrier iterations
solve_result = solved
Time.sstatus = bas

:              Make        Make.sstatus    :=
bands   5999.999925          sup
coils      4.318594312e-05   sup
plate   1600.000006          sup
;



### Another way in amplpy

Read the three runs against each other:

- **Simplex** takes 1 simplex iteration. The statuses form a basis: `bands` is nonbasic at its
  upper bound (`upp`), `coils` at its lower bound (`low`), `plate` is basic (`bas`), and the
  `Time` constraint's logical variable is nonbasic (`upp`).
- **Interior point with crossover** takes 6 barrier iterations and returns the same vertex
  `6000, 0, 1600` with the same basis statuses. Crossover runs by default after `ipm`.
- **Interior point without crossover** takes the same 6 barrier iterations and still reports
  `solved`, but the values lie slightly inside the feasible region (`coils` is `0.000043`, not
  `0`) and the objective is `196399.9996`. No variable is reported at a bound: every `Make` is
  `sup` (superbasic in AMPL's `sstatus_table`) and `Time` is `bas`. These statuses do not form a
  basis, so nothing basis-dependent can be taken from this run.

> **`solved` does not mean "basic"**
> Without crossover the solve status is still `solved`. The difference shows only in the
> statuses and in values that sit within tolerance of, but not on, the vertex. Check `.sstatus`
> before using any basis-dependent output.

**Reuse a crossover basis in the next solve.** A basis from crossover is a full basis, so AMPL
can send it to the next solve, as it does after simplex. With the same `model.mod` and
`data.py`, this run follows each barrier run with a simplex run in the same session, with
`send_statuses` at its default `1`:

In [9]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [10]:
# data.py
ampl.set["PROD"] = ["bands", "coils", "plate"]
ampl.param["rate"] = {"bands": 200, "coils": 140, "plate": 160}
ampl.param["profit"] = {"bands": 25, "coils": 30, "plate": 29}
ampl.param["market"] = {"bands": 6000, "coils": 4000, "plate": 3500}
ampl.param["avail"] = 40

In [11]:
# run.py
for first in ["alg:method=ipm", "alg:method=ipm bar:crossover=off"]:
    ampl.solve(solver="highs", highs_options=first + " pre:solve=off")
    ampl.solve(solver="highs", highs_options="alg:method=simplex pre:solve=off")

HiGHS 1.14.0:   lp:method = ipm
  pre:solve = off
HiGHS 1.14.0: optimal solution; objective 196400
0 simplex iterations
6 barrier iterations
HiGHS 1.14.0:   lp:method = simplex
  pre:solve = off
HiGHS 1.14.0: optimal solution; objective 196400
0 simplex iterations
0 barrier iterations
HiGHS 1.14.0:   lp:method = ipm
  bar:crossover = off
  pre:solve = off
HiGHS 1.14.0: optimal solution; objective 196399.9996
0 simplex iterations
6 barrier iterations
HiGHS 1.14.0:   lp:method = simplex
  pre:solve = off
HiGHS 1.14.0: optimal solution; objective 196400
1 simplex iterations
0 barrier iterations


### Pure AMPL: data.dat and model.run

The same in pure AMPL, with the same `model.mod` and data:

In [12]:
%%writefile data.dat
set PROD := bands coils plate;
param rate := bands 200  coils 140  plate 160;
param profit := bands 25  coils 30  plate 29;
param market := bands 6000  coils 4000  plate 3500;
param avail := 40;

Overwriting data.dat


In [13]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data.dat;

In [14]:
%%ampl_eval
# model.run
option solver highs;
option highs_options 'alg:method=ipm pre:solve=off';
solve;
option highs_options 'alg:method=simplex pre:solve=off';
solve;
option highs_options 'alg:method=ipm bar:crossover=off pre:solve=off';
solve;
option highs_options 'alg:method=simplex pre:solve=off';
solve;

HiGHS 1.14.0:   lp:method = ipm
  pre:solve = off
HiGHS 1.14.0: optimal solution; objective 196400
0 simplex iterations
6 barrier iterations
HiGHS 1.14.0:   lp:method = simplex
  pre:solve = off
HiGHS 1.14.0: optimal solution; objective 196400
0 simplex iterations
0 barrier iterations
HiGHS 1.14.0:   lp:method = ipm
  bar:crossover = off
  pre:solve = off
HiGHS 1.14.0: optimal solution; objective 196399.9996
0 simplex iterations
6 barrier iterations
HiGHS 1.14.0:   lp:method = simplex
  pre:solve = off
HiGHS 1.14.0: optimal solution; objective 196400
1 simplex iterations
0 barrier iterations
